# Phase 3 · S4 — Hybrid ALS⊕chainRec (positive shot) + robustness multi-seed

**Mục tiêu:**
1. **Hybrid rank-fusion ALS ⊕ chainRec** (z-score per user, sweep α) — ALS giỏi CF/popularity, chainRec giỏi
   cấu trúc chuỗi → fusion có thể **vượt cả hai**. Nếu best-hybrid AUC > ALS (0.9646) ⇒ **kết quả DƯƠNG mới**.
2. **Multi-seed** (≥3) khẳng định 2 positive đã có là thật (mean±std): **stagewise > uniform** và **ALS > chainRec**.
3. Bảng tổng kết toàn bộ phase, đánh dấu rõ POSITIVE vs LIMIT.

Universe = interactions (giống S2): cùng `data_test`/pool/mask. chainRec dùng checkpoint S1a; ALS train lại (~60s).
**Nền tảng:** Kaggle GPU T4 + Internet On. Multi-seed train chainRec là phần tốn giờ (gate bằng cờ).


## 0 · Setup

In [ ]:
import os, json, time, pickle, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Literal
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    import implicit
except ImportError:
    os.system("pip install -q implicit"); import implicit
from scipy.sparse import csr_matrix

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| implicit", implicit.__version__)
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception: pass

## 1 · Config + load (interactions universe + chainRec checkpoints)

In [ ]:
HF_REPO="vngclinh/goodreads-preprocessed"
PROC_LOCAL=Path("/kaggle/working/processed"); CKPT_LOCAL=Path("/kaggle/working/chainrec")
S4_LOCAL=Path("/kaggle/working/s4"); S4_LOCAL.mkdir(parents=True, exist_ok=True)
EVAL_USERS=5000; BATCH_USERS=64; K_LIST=(10,20)

from huggingface_hub import hf_hub_download, HfApi
def _hf(rel): return hf_hub_download(HF_REPO, rel, repo_type="dataset", token=HF_TOKEN)
def load_npy(n):
    p=PROC_LOCAL/n; return np.load(p if p.exists() else _hf(f"chainrec/processed/{n}"))
def load_pkl(n):
    p=PROC_LOCAL/n; return pickle.load(open(p if p.exists() else _hf(f"chainrec/processed/{n}"),"rb"))
def load_meta():
    p=PROC_LOCAL/"meta.json"; return json.loads(Path(p if p.exists() else _hf("chainrec/processed/meta.json")).read_text())
def ckpt_path(s,tag=""):
    fn=f"chainrec_{s}{tag}.pt"; p=CKPT_LOCAL/fn
    return str(p) if p.exists() else _hf(f"chainrec/{fn}")

data_train=load_npy("data_train.npy"); data_val=load_npy("data_val.npy"); data_test=load_npy("data_test.npy")
user_item_map=load_pkl("user_item_map.pkl"); meta=load_meta()
N_ITEM,N_USER,N_STAGE=meta["n_item"],meta["n_user"],meta["n_stage"]; REC=N_STAGE-1
s2_base=json.loads(Path(_hf("s2/s2_headtohead.json")).read_text())
print(f"n_user={N_USER:,} n_item={N_ITEM:,} REC={REC}")
print("S2 baselines AUC:", {k:round(v['AUC'],4) for k,v in s2_base.items()})

## 2 · Model + rank_eval + scorers (chainRec / ALS / **hybrid**)

In [ ]:
@dataclass
class ModelConfig:
    n_user:int; n_item:int; n_stage:int=4; embed_dim:int=16
    beta:float=1.0; learn_beta:bool=True; l2:float=0.01; lr:float=0.001
    batch_size:int=2048; n_neg:int=1; n_epochs:int=30; patience:int=5
    sampler:Literal["uniform","stagewise"]="uniform"; device:str=DEVICE
class ChainRecModel(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.cfg=cfg; K,L=cfg.embed_dim,cfg.n_stage
        self.user_emb=nn.Embedding(cfg.n_user,K); self.item_emb=nn.Embedding(cfg.n_item,K)
        self.stage_emb=nn.Embedding(L,K); self.b0=nn.Parameter(torch.zeros(1))
        self.b_user=nn.Embedding(cfg.n_user,1); self.b_item=nn.Embedding(cfg.n_item,1)
        lb=torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta: self.log_beta=nn.Parameter(lb)
        else: self.register_buffer("log_beta",lb)
        for e in [self.user_emb,self.item_emb,self.stage_emb]: nn.init.xavier_uniform_(e.weight)
        for b in [self.b_user,self.b_item]: nn.init.zeros_(b.weight)
    @property
    def beta(self): return torch.clamp(self.log_beta.exp(),min=1.0)
    def _intention(self,u,i,l): return (self.stage_emb(l)*self.item_emb(i)*self.user_emb(u)).sum(-1)
    def _rect(self,d): b=self.beta; return F.softplus(b*d)/b
    def score(self,u,i,ts):
        B=u.shape[0]; bias=self.b0+self.b_user(u).squeeze(-1)+self.b_item(i).squeeze(-1)
        acc=torch.zeros(B,device=u.device)
        for lp in range(ts,self.cfg.n_stage):
            acc=acc+self._rect(self._intention(u,i,torch.full((B,),lp,dtype=torch.long,device=u.device)))
        return bias+acc
    def edgewise_terms(self,u,i,l_star):
        L=self.cfg.n_stage; bias=self.b0+self.b_user(u).squeeze(-1)+self.b_item(i).squeeze(-1)
        dp=torch.stack([self._rect(self._intention(u,i,torch.full((u.shape[0],),l,dtype=torch.long,device=u.device))) for l in range(L)],dim=1)
        suffix=dp.flip(dims=[1]).cumsum(dim=1).flip(dims=[1]); s=bias.unsqueeze(1)+suffix
        lc=l_star.clamp(0,L-1)
        s_l=s.gather(1,lc.unsqueeze(1)).squeeze(1)
        s_n=torch.where(l_star==L-1, torch.full_like(s_l,-1e9), s.gather(1,(l_star+1).clamp(0,L-1).unsqueeze(1)).squeeze(1))
        p_cap=(1.0-torch.exp(-self._rect(self._intention(u,i,lc)))).clamp(min=1e-8)
        return torch.sigmoid(s_l), torch.sigmoid(s_n), p_cap
def build_model(sampler="uniform"):
    cfg=ModelConfig(n_user=N_USER,n_item=N_ITEM,n_stage=N_STAGE,sampler=sampler)
    return ChainRecModel(cfg).to(DEVICE), cfg

@torch.no_grad()
def make_chainrec_scorer(model,ts):
    model.eval(); ie=model.item_emb.weight; bi=model.b_item.weight.squeeze(-1); sw=model.stage_emb.weight
    def fn(u):
        uv=model.user_emb(u); bias=(model.b0+model.b_user(u).squeeze(-1)).unsqueeze(1)
        acc=torch.zeros(u.shape[0],ie.shape[0],device=u.device)
        for l in range(ts,model.cfg.n_stage): acc=acc+model._rect((uv*sw[l].unsqueeze(0))@ie.t())
        return bias+bi.unsqueeze(0)+acc
    return fn
@torch.no_grad()
def make_als_scorer(U,V,device="cuda"):
    Ut=torch.as_tensor(U,dtype=torch.float32,device=device); Vt=torch.as_tensor(V,dtype=torch.float32,device=device)
    def fn(u): return Ut[u]@Vt.t()
    return fn
@torch.no_grad()
def make_hybrid_scorer(als_fn, cr_fn, alpha):
    # rank-fusion bằng z-score theo từng user: alpha*z(ALS) + (1-alpha)*z(chainRec)
    def z(s): return (s-s.mean(1,keepdim=True))/(s.std(1,keepdim=True)+1e-8)
    def fn(u): return alpha*z(als_fn(u)) + (1.0-alpha)*z(cr_fn(u))
    return fn
@torch.no_grad()
def make_pop_scorer(pop,device="cuda"):
    p=torch.as_tensor(pop,dtype=torch.float32,device=device)
    def fn(u): return p.unsqueeze(0).expand(u.shape[0],-1)
    return fn

@torch.no_grad()
def rank_eval(score_fn,test_pairs,user_item_map,n_item,pos_stage=None,K_list=(10,20),
              batch_users=64,n_eval_users=None,device="cuda",seed=999):
    pairs=test_pairs if pos_stage is None else test_pairs[test_pairs[:,2]==pos_stage]
    if n_eval_users is not None and len(pairs)>n_eval_users:
        rng=np.random.default_rng(seed); pairs=pairs[rng.choice(len(pairs),size=n_eval_users,replace=False)]
    aucs=[]; hits={k:[] for k in K_list}; ndcg={k:[] for k in K_list}
    for st in range(0,len(pairs),batch_users):
        ch=pairs[st:st+batch_users]
        u=torch.tensor(ch[:,0],dtype=torch.long,device=device); pos=torch.tensor(ch[:,1],dtype=torch.long,device=device)
        sc=score_fn(u); B=u.shape[0]; ar=torch.arange(B,device=device)
        ps=sc[ar,pos].clone(); seen_cnt=torch.zeros(B,device=device)
        for b in range(B):
            seen=user_item_map.get(int(u[b]),())
            if seen:
                idx=torch.tensor(list(seen),dtype=torch.long,device=device); sc[b,idx]=float("-inf"); seen_cnt[b]=len(seen)
        sc[ar,pos]=ps; rank=(sc>ps.unsqueeze(1)).sum(1).float(); neg=(n_item-seen_cnt).clamp(min=1)
        aucs.append((1.0-rank/neg).cpu().numpy()); rnp=rank.cpu().numpy()
        for k in K_list:
            hit=rnp<k; hits[k].append(hit.astype(float)); ndcg[k].append(np.where(hit,1.0/np.log2(rnp+2),0.0))
        del sc
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    res={"AUC":float(np.concatenate(aucs).mean()),"n_eval":int(len(pairs))}
    for k in K_list:
        res[f"Recall@{k}"]=float(np.concatenate(hits[k]).mean()); res[f"NDCG@{k}"]=float(np.concatenate(ndcg[k]).mean())
    return res

## 3 · Retrain ALS (interactions, vanilla — ~60s) + load chainRec ckpt stagewise

In [ ]:
def train_als(seed=SEED):
    pos=data_train[data_train[:,2]==REC]
    m=csr_matrix((np.ones(len(pos),dtype=np.float32),(pos[:,0].astype(np.int32),pos[:,1].astype(np.int32))),shape=(N_USER,N_ITEM))
    try: a=implicit.als.AlternatingLeastSquares(factors=64,iterations=20,regularization=0.1,alpha=40,random_state=seed,use_gpu=False)
    except TypeError: a=implicit.als.AlternatingLeastSquares(factors=64,iterations=20,regularization=0.1,random_state=seed,use_gpu=False); m=m*40.0
    a.fit(m); return np.asarray(a.user_factors,dtype=np.float32), np.asarray(a.item_factors,dtype=np.float32)

t0=time.time(); U,V=train_als(); print(f"ALS fit {time.time()-t0:.1f}s")
als_fn=make_als_scorer(U,V,device=DEVICE)

m_cr,_=build_model("stagewise")
try: m_cr.load_state_dict(torch.load(ckpt_path("stagewise","_r10"),map_location=DEVICE))
except Exception: m_cr.load_state_dict(torch.load(ckpt_path("stagewise"),map_location=DEVICE)); print("(dùng ckpt S0 stagewise)")
cr_fn=make_chainrec_scorer(m_cr, REC)
print("Loaded ALS + chainRec(stagewise).")

## 4 · **Hybrid α-sweep** (positive shot)

α=1 → ALS thuần · α=0 → chainRec thuần · 0<α<1 → fusion. Nếu best-hybrid AUC > ALS ⇒ DƯƠNG.

In [ ]:
ALS_AUC=s2_base["ALS(vanilla)"]["AUC"]; CR_AUC=s2_base["chainRec(stagewise)"]["AUC"]
hyb={}
for a in [0.0,0.25,0.4,0.5,0.6,0.75,1.0]:
    r=rank_eval(make_hybrid_scorer(als_fn,cr_fn,a), data_test, user_item_map, N_ITEM,
                pos_stage=REC, K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
    hyb[a]=r; print(f"α={a:.2f}  AUC={r['AUC']:.4f}  R@10={r['Recall@10']:.4f}  N@10={r['NDCG@10']:.4f}")
best_a=max(hyb,key=lambda k:hyb[k]['AUC']); best=hyb[best_a]
print(f"\nbest α={best_a:.2f}  AUC={best['AUC']:.4f}  R@10={best['Recall@10']:.4f}")
print(f"baseline ALS={ALS_AUC:.4f}  chainRec={CR_AUC:.4f}")
win = best['AUC']>max(ALS_AUC,CR_AUC)+1e-4
print(f">>> Hybrid {'VƯỢT' if win else 'KHÔNG vượt'} cả hai baseline "
      f"(Δ vs ALS={best['AUC']-ALS_AUC:+.4f}, {(best['AUC']/ALS_AUC-1)*100:+.2f}%)")
json.dump({str(k):v for k,v in hyb.items()}, open(S4_LOCAL/"s4_hybrid_sweep.json","w"), indent=2)

## 5 · Multi-seed (robustness): stagewise vs uniform, ALS — mean±std

`RUN_MULTISEED=True` train chainRec `N_SEEDS` lần/ sampler (TỐN GIỜ ~30-45'/seed). Đặt False để bỏ qua.

In [ ]:
def edgewise_loss(model,up,ip,lp,un,ing,ln,l2,w_pos=None):
    p_pos,_,_=model.edgewise_terms(up,ip,lp); _,p_n,p_cap=model.edgewise_terms(un,ing,ln)
    lpos=torch.log(p_pos.clamp(min=1e-8)); loss_pos=-(w_pos*lpos).mean() if w_pos is not None else -lpos.mean()
    loss_neg=-(torch.log((1-p_n).clamp(min=1e-8))+torch.log(p_cap)).mean()
    l2_loss=l2*(model.user_emb.weight.norm(2)**2+model.item_emb.weight.norm(2)**2)/(model.cfg.n_user+model.cfg.n_item)
    return loss_pos+loss_neg+l2_loss
class CData(Dataset):
    def __init__(self,data,n_item,seed): self.data=data; self.n_item=n_item; self.rng=np.random.default_rng(seed)
    def __len__(self): return len(self.data)
    def __getitem__(self,idx):
        u,i,l=[int(x) for x in self.data[idx]]; return (u,i,l,u,int(self.rng.integers(0,self.n_item)),l)

# monitor NHANH: chỉ chấm trên <=MON_USERS rec-edge test (đủ để early-stop)
MON_USERS=1500
@torch.no_grad()
def sampled_recall(model,dt,uim,n_item,ts,n_neg=500,k=10,device="cuda",max_users=MON_USERS):
    model.eval(); rng=np.random.default_rng(999)
    rec=dt[dt[:,2]==ts]
    if len(rec)>max_users: rec=rec[rng.choice(len(rec),size=max_users,replace=False)]
    hits=[]
    for u,ipos,l in rec:
        u,ipos=int(u),int(ipos); pos=uim.get(u,set()); negs=[]; t=0
        while len(negs)<n_neg and t<n_neg*5:
            c=rng.integers(0,n_item)
            if c not in pos and c!=ipos: negs.append(c)
            t+=1
        items=np.array([ipos]+negs)
        sc=model.score(torch.full((len(items),),u,dtype=torch.long,device=device),torch.tensor(items,dtype=torch.long,device=device),ts).cpu().numpy()
        hits.append(int(int(np.where(np.argsort(-sc)==0)[0][0])<k))
    return float(np.mean(hits))

def train_cr(sampler,seed,n_epochs=15,patience=3):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model,cfg=build_model(sampler); opt=torch.optim.Adam(model.parameters(),lr=cfg.lr)
    dl=DataLoader(CData(data_train,N_ITEM,seed),batch_size=cfg.batch_size,shuffle=True,num_workers=4,pin_memory=True)
    best,pc,save=-1.0,0,str(S4_LOCAL/f"cr_{sampler}_s{seed}.pt")
    print(f"   {'ep':>3} | {'loss':>8} | {'R@10(s)':>8} | {'sec':>5}")
    for ep in range(1,n_epochs+1):
        model.train(); t0=time.time(); tot=nb=0
        for u,i,l,un,ni,ln in dl:
            u,i,l=u.to(DEVICE),i.to(DEVICE),l.to(DEVICE); un,ni,ln=un.to(DEVICE),ni.to(DEVICE),ln.to(DEVICE)
            opt.zero_grad(); loss=edgewise_loss(model,u,i,l,un,ni,ln,cfg.l2)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); tot+=loss.item(); nb+=1
        r10=sampled_recall(model,data_test,user_item_map,N_ITEM,REC,device=DEVICE)
        print(f"   {ep:>3} | {tot/nb:>8.4f} | {r10:>8.4f} | {time.time()-t0:>5.0f}")
        if r10>best: best,pc=r10,0; torch.save(model.state_dict(),save)
        else:
            pc+=1
            if pc>=patience: print(f"   early stop @ {ep} (best R@10={best:.4f})"); break
    model.load_state_dict(torch.load(save,map_location=DEVICE)); return model

RUN_MULTISEED=True; N_SEEDS=3; SEEDS=[42,1,2][:N_SEEDS]
SAMPLERS_MS=["uniform","stagewise"]   # bỏ "uniform" nếu muốn nhanh gấp đôi (chỉ cần stagewise mean±std)
ms={}
if RUN_MULTISEED:
    n_train=len(SAMPLERS_MS)*N_SEEDS
    print(f"Sẽ train {n_train} lần chainRec (~10-20'/lần) + {N_SEEDS} ALS. Ước lượng ~{n_train*15} phút.\n")
    for sampler in SAMPLERS_MS:
        accs=[]
        for sd in SEEDS:
            print(f"train chainRec {sampler} seed={sd} ...")
            m=train_cr(sampler,sd)
            r=rank_eval(make_chainrec_scorer(m,REC),data_test,user_item_map,N_ITEM,pos_stage=REC,
                        K_list=K_LIST,batch_users=BATCH_USERS,n_eval_users=EVAL_USERS,device=DEVICE)
            accs.append(r["AUC"]); print(f"   -> full-rank AUC={r['AUC']:.4f}\n")
        ms[sampler]={"AUC_mean":float(np.mean(accs)),"AUC_std":float(np.std(accs)),"seeds":accs}
    als_accs=[]
    for sd in SEEDS:
        Us,Vs=train_als(sd)
        r=rank_eval(make_als_scorer(Us,Vs,device=DEVICE),data_test,user_item_map,N_ITEM,pos_stage=REC,
                    K_list=K_LIST,batch_users=BATCH_USERS,n_eval_users=EVAL_USERS,device=DEVICE)
        als_accs.append(r["AUC"])
    ms["ALS"]={"AUC_mean":float(np.mean(als_accs)),"AUC_std":float(np.std(als_accs)),"seeds":als_accs}
    print("=== multi-seed AUC (mean±std) ===")
    for k,v in ms.items(): print(f"{k:10} {v['AUC_mean']:.4f} ± {v['AUC_std']:.4f}  {[round(x,4) for x in v['seeds']]}")
    json.dump(ms, open(S4_LOCAL/"s4_multiseed.json","w"), indent=2)
else:
    print("RUN_MULTISEED=False — bỏ qua phần train tốn giờ.")

## 6 · Bảng tổng kết toàn phase (đánh dấu POSITIVE vs LIMIT)

In [ ]:
print("="*64)
print("TỔNG KẾT — Goodreads recommender (Phase 2 + Phase 3)")
print("="*64)
print("\n[POSITIVE] Phase 2 ablation @ sampled@500:")
print("   F3(length) R@10=0.6122 > F2 baseline 0.5757  (+3.6%)  ← engagement-weight thắng")
print("\n[POSITIVE] stagewise > uniform (chainRec):")
if "stagewise" in ms:
    print(f"   AUC stagewise {ms['stagewise']['AUC_mean']:.4f}±{ms['stagewise']['AUC_std']:.4f}"
          f"  >  uniform {ms['uniform']['AUC_mean']:.4f}±{ms['uniform']['AUC_std']:.4f}  (full-rank, {N_SEEDS} seeds)")
else:
    print("   AUC 0.9525 > 0.9500 (single-seed S2)  [bật RUN_MULTISEED cho mean±std]")
print(f"\n[{'POSITIVE' if win else 'NEUTRAL'}] Hybrid ALS⊕chainRec:")
print(f"   best α={best_a:.2f} AUC={best['AUC']:.4f}  vs ALS {ALS_AUC:.4f}  ({(best['AUC']/ALS_AUC-1)*100:+.2f}%)")
print("\n[LIMIT] F3 dưới full-ranking (S3/S3b): −0.8% AUC; coverage interactions 5.5%  → không transfer sang chainRec")
print("[CONFIRM] votes gây popularity bias: −5.8% AUC (S3b), −17.6% R@10 (Phase 2)")
print(f"[BASELINE] ALS thuần > chainRec trên Goodreads full-ranking ({ALS_AUC:.4f} > {CR_AUC:.4f})")

summary={"hybrid_best_alpha":best_a,"hybrid":best,"als_auc":ALS_AUC,"chainrec_auc":CR_AUC,
         "hybrid_wins":bool(win),"multiseed":ms}
json.dump(summary, open(S4_LOCAL/"s4_summary.json","w"), indent=2)
print("\nSaved s4_summary.json")

## 7 · Push

In [ ]:
to_push=sorted(S4_LOCAL.glob("*.json"))
if HF_TOKEN and to_push:
    api=HfApi()
    for p in to_push:
        api.upload_file(path_or_fileobj=str(p),path_in_repo=f"s4/{p.name}",repo_id=HF_REPO,repo_type="dataset",token=HF_TOKEN)
        print("  ✓",p.name)
else:
    print("Không push. Artifact ở /kaggle/working/s4.")

## 8 · Ghi chú báo cáo

- **Nếu hybrid VƯỢT ALS** → đây là positive result chính, mới: "fusion chuỗi-hành-vi + CF vượt từng mô hình".
- **Nếu hybrid không vượt** → vẫn còn 2 positive chắc chắn: (a) F3 thắng ablation @ sampled@500 (Phase 2),
  (b) stagewise > uniform (tái lập paper, mean±std). Báo cáo nêu cả hai là đóng góp dương, kèm phân tích
  full-ranking như "đánh giá nghiêm ngặt vạch giới hạn".
- Hybrid z-score fusion là parameter-light, dễ giải thích — phù hợp mục method.
